# 02f — SVM-RBF operating-point (decision-threshold) tuning

**CAC2026 Data Challenge — finalising the chosen model.**

The classifier sweep (02a–02e) is settled: every head clusters at ~0.76–0.78
nested F1, the ceiling is feature-bound, and **SVM-RBF was locked** as the
carried-forward model for its low variance and clean decision score. Its
hyperparameters are fixed from 02d:

```
block LVs (4, 9, 9) · final SVC(kernel='rbf', C=1, gamma='scale') · scale_fused=True
```

Every notebook so far used the **default 0.5 decision threshold**. Under ~34 %
musty prevalence the F1-optimal threshold is rarely 0.5, so here we tune the
operating point with `TunedThresholdClassifierCV` (sklearn's CV-based threshold
optimiser, scoring F1). This is the one remaining lever that *adds* information
(it uses the probability scores we already compute) and directly targets the
competition metric.

> **Expectation set in advance:** on the held-out test set only ~1 prediction sits
> near the boundary, so threshold tuning is unlikely to swing the submission much.
> The honest question is whether the F1-optimal threshold differs from 0.5 enough
> to matter — we estimate that without leakage below.

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.model_selection import TunedThresholdClassifierCV, cross_validate
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

warnings.filterwarnings("ignore")

from olive_oil import (
    load_dataset, BlockConfig, prepare_blocks,
    MidLevelFusionClassifier, predict_test, make_cv,
)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "Notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "Data" / "CAC2026_Data_challenge.xlsx"

raw = load_dataset(DATA_PATH)
configs = {
    "hsms": BlockConfig(region=(100, 125),
                        steps=["row_profile", "log", "mean_center"]),
    "mir": BlockConfig(region=(1500, 700),
                       steps=["snv", ("savgol_derivative", {"deriv": 1, "polyorder": 2, "half_window": 6}),
                              "mean_center"]),
    "uvvis": BlockConfig(region=(300, 1000),
                         steps=["snv", ("savgol_derivative", {"deriv": 1, "polyorder": 2, "half_window": 6}),
                                "mean_center"]),
}
data = prepare_blocks(raw, configs, outliers=[])
print("Calibration:", len(data.y), "| musty:", int(data.y.sum()))
data.summary()

Calibration: 220 | musty: 76


,n_variables,axis_min,axis_max,n_train,n_test
block,,,,,
hsms,26,100.00000,125.000000,220,24
mir,208,700.04498,1498.443334,220,24
uvvis,701,300.00000,1000.000000,220,24


## 1. The locked SVM model

Fixed hyperparameters from 02d — nothing is searched here.

In [2]:
model = MidLevelFusionClassifier(
    n_components_list=(4, 9, 9),
    classifier=SVC(kernel="rbf", C=1, gamma="scale", probability=True, random_state=0),
    scale_fused=True,
)

SCORING = {"f1": "f1", "precision": "precision", "recall": "recall"}
outer_cv = make_cv(5, random_state=0)

## 2. Baseline — honest CV at the default 0.5 threshold

With hyperparameters fixed, a plain 5-fold `cross_validate` is already an honest
estimate (no inner model selection to nest). This reproduces the 02d SVM number
and is the baseline the tuned threshold must beat.

In [3]:
base = cross_validate(model, data.train_blocks(), data.y,
                      cv=outer_cv, scoring=SCORING, return_train_score=False)
base_summary = pd.Series({m: base[f"test_{m}"].mean() for m in SCORING}, name="threshold=0.5")
print("Baseline (default 0.5 threshold) — outer-fold means:")
print(base_summary.round(3))
print("Per-fold F1:", np.round(base["test_f1"], 3),
      f"-> {base['test_f1'].mean():.3f} +/- {base['test_f1'].std():.3f}")

Baseline (default 0.5 threshold) — outer-fold means:
f1           0.792
precision    0.838
recall       0.751
Name: threshold=0.5, dtype: float64
Per-fold F1: [0.786 0.828 0.828 0.786 0.733] -> 0.792 +/- 0.035


## 3. F1-tuned threshold — honest nested CV

Each outer fold fits a `TunedThresholdClassifierCV` that picks the F1-optimal
threshold by **inner** CV on that fold's training data only, then scores the
held-out outer fold. No test information leaks into the threshold choice, so the
outer mean is a fair estimate of "SVM + tuned threshold".

In [4]:
tuned_clf = TunedThresholdClassifierCV(
    model, scoring="f1", response_method="predict_proba",
    cv=make_cv(5, random_state=1), random_state=0,
)

tuned = cross_validate(tuned_clf, data.train_blocks(), data.y,
                       cv=outer_cv, scoring=SCORING, return_estimator=True)
tuned_summary = pd.Series({m: tuned[f"test_{m}"].mean() for m in SCORING}, name="tuned threshold")

fold_thresholds = [est.best_threshold_ for est in tuned["estimator"]]
print("Tuned threshold — outer-fold means:")
print(tuned_summary.round(3))
print("Per-fold F1:", np.round(tuned["test_f1"], 3),
      f"-> {tuned['test_f1'].mean():.3f} +/- {tuned['test_f1'].std():.3f}")
print("Per-fold chosen thresholds:", [round(t, 3) for t in fold_thresholds])

Tuned threshold — outer-fold means:
f1           0.749
precision    0.729
recall       0.778
Name: tuned threshold, dtype: float64
Per-fold F1: [0.8   0.727 0.839 0.647 0.733] -> 0.749 +/- 0.066
Per-fold chosen thresholds: [np.float64(0.385), np.float64(0.314), np.float64(0.425), np.float64(0.223), np.float64(0.515)]


## 4. Does tuning help? (honest comparison)

In [5]:
compare = pd.concat([base_summary, tuned_summary], axis=1)
compare.loc["—"] = ["", ""]
print("Honest nested-CV comparison (default 0.5 vs F1-tuned threshold):")
display(compare.round(3))

d = tuned_summary["f1"] - base_summary["f1"]
print(f"F1 change from threshold tuning: {d:+.3f}")
print("Verdict:", "tuning helps" if d > 0.005 else
      ("tuning neutral (within noise) — 0.5 already near-optimal" if abs(d) <= 0.005
       else "tuning hurts — keep 0.5"))

Honest nested-CV comparison (default 0.5 vs F1-tuned threshold):


,threshold=0.5,tuned threshold
f1,0.791987,0.749275
precision,0.838462,0.728766
recall,0.750833,0.7775
—,,


F1 change from threshold tuning: -0.043
Verdict: tuning hurts — keep 0.5


## 5. Deploy the honest winner

Deploy the tuned threshold **only if** it beat the default 0.5 in the honest §4
comparison; otherwise keep 0.5. Here tuning loses (it overfits an unstable
per-fold threshold), so the deployed model is the plain SVM at 0.5.

In [6]:
USE_TUNED = (tuned_summary["f1"] - base_summary["f1"]) > 0.005

if USE_TUNED:
    final = TunedThresholdClassifierCV(
        model, scoring="f1", response_method="predict_proba",
        cv=make_cv(5, random_state=1), random_state=0, refit=True,
    ).fit(data.train_blocks(), data.y)
    deployed_threshold = float(final.best_threshold_)
    print(f"Threshold tuning helped -> deploying tuned threshold {deployed_threshold:.3f}")
else:
    final = model.fit(data.train_blocks(), data.y)   # plain SVM, default 0.5
    deployed_threshold = 0.5
    print(f"Threshold tuning did NOT help (honest F1 {tuned_summary['f1']:.3f} "
          f"<= {base_summary['f1']:.3f}) -> deploying default threshold 0.5")

y_tr = final.predict(data.train_blocks())
train_scores = pd.DataFrame({"train": [
    f1_score(data.y, y_tr), precision_score(data.y, y_tr),
    recall_score(data.y, y_tr), 100.0 * (1.0 - accuracy_score(data.y, y_tr)),
]}, index=["F1", "Precision", "Recall", "Inaccuracy (%)"])
print(f"Final deployed model train-set fit (threshold = {deployed_threshold:.3f}):")
display(train_scores.round(3))

Threshold tuning did NOT help (honest F1 0.749 <= 0.792) -> deploying default threshold 0.5
Final deployed model train-set fit (threshold = 0.500):


,train
F1,0.939
Precision,0.972
Recall,0.908
Inaccuracy (%),4.091


## 6. Predict the test set and compare to the 0.5 baseline

Save the final submission, and show which test samples (if any) differ from the
default-0.5 SVM (`predictions_02d_SVM.csv`). If 0.5 was deployed, the two match.

In [7]:
predictions = predict_test(final, data.test_blocks(), data.test_ids)
out_path = PROJECT_ROOT / "Data" / "predictions_02f_SVM_final.csv"
predictions.to_csv(out_path, index=False)
print(f"Deployed threshold: {deployed_threshold:.3f}")
print("Predicted test balance:",
      predictions["prediction"].value_counts().sort_index().to_dict())
print("Saved:", out_path)

# Compare to the default-0.5 SVM submission from 02d.
base_path = PROJECT_ROOT / "Data" / "predictions_02d_SVM.csv"
if base_path.exists():
    base_pred = pd.read_csv(base_path)[["sample_id", "prediction"]].rename(
        columns={"prediction": "pred_0.5"})
    merged = predictions.rename(columns={"prediction": "pred_tuned"}).merge(
        base_pred.astype({"sample_id": predictions["sample_id"].dtype}), on="sample_id")
    flipped = merged[merged["pred_tuned"] != merged["pred_0.5"]]
    print(f"\nFlipped vs default 0.5: {len(flipped)} / {len(merged)} samples")
    if len(flipped):
        display(flipped[["sample_id", "pred_0.5", "pred_tuned", "proba_musty"]].round(3))
predictions

Deployed threshold: 0.500
Predicted test balance: {0: 17, 1: 7}
Saved: C:\Users\SamdGuizani\OneDrive\Documents\Data Science & Coding\2026-Olive Oil Classification (CAC 2026)\CAC-2026_Data-Challenge_Olive-Oil-Sensory-Defects-Detection-Using-Chemometrics\Data\predictions_02f_SVM_final.csv

Flipped vs default 0.5: 0 / 24 samples


,sample_id,prediction,proba_musty
0,1,1,0.994793
1,2,0,0.007994
2,3,1,0.941378
3,4,0,0.099572
4,5,0,0.011647
5,6,0,0.008780
6,7,0,0.028324
7,8,1,0.724206
8,9,0,0.157373
9,10,0,0.029745


## Summary

- **Model locked:** SVM-RBF on fused PLS scores — `(4, 9, 9)`, `C=1`,
  `gamma='scale'`, `scale_fused=True`.
- **Operating point:** F1-threshold tuning was evaluated honestly (§4) and
  **rejected** — the per-fold optimal threshold is unstable at n = 220, so tuning
  it overfits and *lowers* the honest F1 (0.79 → 0.75). The deployed model keeps
  the **default 0.5 threshold**.
- This is the **final submission pipeline** for the SVM track (≈ 0.79 honest F1 at
  threshold 0.5). The remaining upside, if any, is in the **features**
  (preprocessing / regions), per the classifier-sweep conclusion — not in the
  classifier or its threshold.